# Python Functions — A Scientific Computing Tutorial
### Functions · Closures · Decorators · Generators · NumPy · SciPy · Toxicology Applications

**Author:** Himanshu Goel | [himanshugoel.github.io](https://himanshugoel.github.io)

---

This notebook teaches Python functions **through scientific computing examples** drawn from
computational toxicology, cheminformatics, and data analysis. Every concept is grounded
in something you will actually use in research.

| Section | Topics |
|---------|--------|
| 1. Function basics | def, args, kwargs, defaults, docstrings |
| 2. First-class functions | Passing functions, map/filter/reduce |
| 3. Lambda & comprehensions | Inline transforms on molecular datasets |
| 4. *args and **kwargs | Flexible descriptor calculators |
| 5. Closures | Stateful functions, factory pattern |
| 6. Decorators | Timing, caching, validation wrappers |
| 7. Generators & iterators | Memory-efficient library processing |
| 8. Recursive functions | Scaffold decomposition, combinatorics |
| 9. Functional programming | Pipelines for QSAR preprocessing |
| 10. NumPy vectorised functions | Broadcasting, ufuncs, apply_along_axis |
| 11. SciPy for toxicology | Dose-response, statistics, optimisation |
| 12. Error handling | try/except, custom exceptions, safe wrappers |

---
## Section 1 — Function Basics

In [ ]:
# ── 1.1 Anatomy of a Python function ─────────────────────────────────────────
# Every scientific function should have: docstring, type hints, return value

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, QED
import numpy as np
import pandas as pd

def compute_mw(smiles: str) -> float | None:
    """
    Compute molecular weight from a SMILES string.

    Parameters
    ----------
    smiles : str
        SMILES representation of the molecule.

    Returns
    -------
    float or None
        Molecular weight in Da, or None if SMILES is invalid.

    Examples
    --------
    >>> compute_mw("CC(=O)Oc1ccccc1C(=O)O")
    180.158   # Aspirin
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return round(Descriptors.MolWt(mol), 3)

# Test it
print(compute_mw("CC(=O)Oc1ccccc1C(=O)O"))      # Aspirin: 180.158
print(compute_mw("Cn1cnc2c1c(=O)n(C)c(=O)n2C")) # Caffeine
print(compute_mw("INVALID"))                      # None

In [ ]:
# ── 1.2 Default arguments and keyword arguments ──────────────────────────────
def lipinski_screen(smiles: str,
                    mw_limit: float = 500.0,
                    logp_limit: float = 5.0,
                    hbd_limit: int = 5,
                    hba_limit: int = 10) -> dict:
    """
    Apply Lipinski Rule of Five screening.
    All limits are keyword arguments with standard defaults.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {"valid": False}

    mw   = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd  = rdMolDescriptors.CalcNumHBD(mol)
    hba  = rdMolDescriptors.CalcNumHBA(mol)

    violations = sum([mw > mw_limit, logp > logp_limit,
                      hbd > hbd_limit, hba > hba_limit])
    return {
        "valid": True, "MW": round(mw,1), "LogP": round(logp,2),
        "HBD": hbd, "HBA": hba,
        "violations": violations,
        "passes_ro5": violations <= 1
    }

# Standard call
print(lipinski_screen("CC(=O)Oc1ccccc1C(=O)O"))

# Tighter limits for CNS drugs (keyword args override defaults)
print(lipinski_screen("Cn1cnc2c1c(=O)n(C)c(=O)n2C",
                      mw_limit=400, logp_limit=3, hbd_limit=3))

In [ ]:
# ── 1.3 Multiple return values ────────────────────────────────────────────────
# Python returns tuples — unpack at the call site

def admet_profile(smiles: str) -> tuple[float, float, float, int, int] | None:
    """Return (MW, LogP, TPSA, HBD, HBA) or None."""    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return (
        round(Descriptors.MolWt(mol), 2),
        round(Descriptors.MolLogP(mol), 3),
        round(Descriptors.TPSA(mol), 2),
        rdMolDescriptors.CalcNumHBD(mol),
        rdMolDescriptors.CalcNumHBA(mol),
    )

# Unpack tuple
mw, logp, tpsa, hbd, hba = admet_profile("CC(=O)Oc1ccccc1C(=O)O")
print(f"MW={mw}  LogP={logp}  TPSA={tpsa}  HBD={hbd}  HBA={hba}")

# Can also capture as named tuple for clarity
from collections import namedtuple
ADMET = namedtuple("ADMET", ["MW", "LogP", "TPSA", "HBD", "HBA"])

def admet_named(smiles: str) -> ADMET | None:
    result = admet_profile(smiles)
    return ADMET(*result) if result else None

profile = admet_named("CC(=O)Nc1ccc(O)cc1")
print(f"Paracetamol TPSA = {profile.TPSA}")  # named access

---
## Section 2 — First-Class Functions

In [ ]:
# ── 2.1 Functions as arguments (higher-order functions) ───────────────────────
# In Python, functions are objects — pass them as arguments just like data.

def screen_library(smiles_list: list[str],
                   scoring_fn,          # any callable
                   threshold: float = 0.5) -> list[str]:
    """
    Filter a compound library by applying scoring_fn to each SMILES.
    Returns SMILES where score >= threshold.
    """
    passing = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        score = scoring_fn(mol)
        if score is not None and score >= threshold:
            passing.append(smi)
    return passing

# Three different scoring functions — all have the same signature: mol → float
def qed_score(mol):
    return QED.qed(mol)

def bbb_score(mol):
    """Rough BBB score: LogP in (1-3), TPSA < 90, MW < 400."""    logp = Descriptors.MolLogP(mol)
    tpsa = Descriptors.TPSA(mol)
    mw   = Descriptors.MolWt(mol)
    return 1.0 if (1 <= logp <= 3 and tpsa < 90 and mw < 400) else 0.0

def solubility_score(mol):
    """Estimate LogS ~ -0.74*LogP + 0.0091*MW - 4.8 (Egan 2000 simplified)."""    logp = Descriptors.MolLogP(mol)
    mw   = Descriptors.MolWt(mol)
    logs = -0.74*logp + 0.0091*mw - 4.8
    # Normalise to [0, 1]: logs > -4 is acceptable
    return max(0, min(1, (logs + 6) / 4))

library = [
    "CC(=O)Oc1ccccc1C(=O)O", "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O", "CC(=O)Nc1ccc(O)cc1",
    "NCCc1ccc(O)c(O)c1", "CCCC1=NN(C)C(=O)c2cc(cnc21)c1ccc(cc1)S(=O)(=O)N1CCN(C)CC1",
]

print("QED ≥ 0.5:        ", screen_library(library, qed_score, 0.5))
print("BBB penetrant:    ", screen_library(library, bbb_score, 0.5))
print("Good solubility:  ", screen_library(library, solubility_score, 0.4)[:2], "...")

In [ ]:
# ── 2.2 map(), filter(), zip(), sorted() with functions ───────────────────────
smiles_list = [
    "CC(=O)Oc1ccccc1C(=O)O", "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O", "CC(=O)Nc1ccc(O)cc1",
    "INVALID_SMILES",
]

# map(): apply function to every element
mols = list(map(Chem.MolFromSmiles, smiles_list))        # SMILES → Mol objects
print(f"map() → {len(mols)} Mol objects ({sum(m is not None for m in mols)} valid)")

# filter(): keep elements matching a condition
valid_mols = list(filter(lambda m: m is not None, mols))
print(f"filter() → {len(valid_mols)} valid molecules")

# zip(): pair two lists together (SMILES + descriptors)
names = ["Aspirin", "Caffeine", "Ibuprofen", "Paracetamol"]
mws   = [Descriptors.MolWt(m) for m in valid_mols]
for name, mw in zip(names, mws):
    print(f"  {name:12s}: MW = {mw:.1f}")

# sorted() with key= function
by_mw = sorted(zip(names, mws), key=lambda pair: pair[1])
print("\nSorted by MW:", [(n, round(mw,1)) for n, mw in by_mw])

---
## Section 3 — Lambda and Comprehensions

In [ ]:
# ── 3.1 Lambda: anonymous functions for one-liners ───────────────────────────
# Use lambdas for short, throwaway functions — never for complex logic

# Sort compounds by QED score
mols_smiles = [
    ("Aspirin",      "CC(=O)Oc1ccccc1C(=O)O"),
    ("Caffeine",     "Cn1cnc2c1c(=O)n(C)c(=O)n2C"),
    ("Ibuprofen",    "CC(C)Cc1ccc(cc1)C(C)C(=O)O"),
    ("Paracetamol",  "CC(=O)Nc1ccc(O)cc1"),
    ("Sildenafil",   "CCCC1=NN(C)C(=O)c2cc(cnc21)c1ccc(cc1)S(=O)(=O)N1CCN(C)CC1"),
]

# Lambda as key in sorted
ranked = sorted(
    [(n, s) for n, s in mols_smiles],
    key=lambda pair: QED.qed(Chem.MolFromSmiles(pair[1])),
    reverse=True
)
print("Ranked by QED (highest first):")
for name, smi in ranked:
    mol = Chem.MolFromSmiles(smi)
    print(f"  {name:12s}: QED = {QED.qed(mol):.4f}")

In [ ]:
# ── 3.2 List, dict and set comprehensions ────────────────────────────────────
smiles_list = [
    "CC(=O)Oc1ccccc1C(=O)O", "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O", "CC(=O)Nc1ccc(O)cc1",
    "INVALID", "NCCc1ccc(O)c(O)c1",
]

# List comprehension: compute descriptor for each valid molecule
mws = [
    Descriptors.MolWt(mol)
    for smi in smiles_list
    if (mol := Chem.MolFromSmiles(smi)) is not None   # walrus operator (:=)
]
print("Molecular weights:", [round(mw, 1) for mw in mws])

# Dict comprehension: SMILES → descriptor dict
desc_dict = {
    smi: {
        "MW":   round(Descriptors.MolWt(mol), 1),
        "LogP": round(Descriptors.MolLogP(mol), 2),
        "QED":  round(QED.qed(mol), 3)
    }
    for smi in smiles_list
    if (mol := Chem.MolFromSmiles(smi)) is not None
}
print(f"\nDescriptor dict for {len(desc_dict)} molecules:")
for smi, desc in list(desc_dict.items())[:2]:
    print(f"  {smi[:20]}...: {desc}")

# Set comprehension: unique scaffolds
from rdkit.Chem.Scaffolds import MurckoScaffold
scaffolds = {
    Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
    for smi in smiles_list
    if (mol := Chem.MolFromSmiles(smi)) is not None
}
print(f"\nUnique Murcko scaffolds: {len(scaffolds)}")

---
## Section 4 — *args and **kwargs

In [ ]:
# ── 4.1 *args: variable positional arguments ─────────────────────────────────
def tanimoto_similarity(*smiles_list: str) -> np.ndarray:
    """
    Compute Tanimoto similarity matrix for any number of SMILES.
    *args lets you pass 2, 5, 100 compounds — same function.
    """
    from rdkit.Chem import AllChem
    from rdkit import DataStructs

    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            fps.append(AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048))

    n = len(fps)
    matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            matrix[i, j] = DataStructs.TanimotoSimilarity(fps[i], fps[j])
    return matrix

# Two compounds
mat2 = tanimoto_similarity("CC(=O)Oc1ccccc1C(=O)O", "CC(=O)Nc1ccc(O)cc1")
print(f"2×2 matrix:\n{mat2.round(3)}")

# Four compounds — same function
mat4 = tanimoto_similarity(
    "CC(=O)Oc1ccccc1C(=O)O", "CC(=O)Nc1ccc(O)cc1",
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O", "Cn1cnc2c1c(=O)n(C)c(=O)n2C"
)
print(f"\n4×4 matrix:\n{mat4.round(3)}")

In [ ]:
# ── 4.2 **kwargs: variable keyword arguments ─────────────────────────────────
def compute_descriptors(smiles: str, **kwargs) -> dict:
    """
    Compute descriptors. Pass which ones you want as keyword flags.

    Usage:
        compute_descriptors(smi, mw=True, logp=True, tpsa=True)
        compute_descriptors(smi, all=True)
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {}

    compute_all = kwargs.get("all", False)
    result = {}

    if compute_all or kwargs.get("mw"):
        result["MW"]       = round(Descriptors.MolWt(mol), 2)
    if compute_all or kwargs.get("logp"):
        result["LogP"]     = round(Descriptors.MolLogP(mol), 3)
    if compute_all or kwargs.get("tpsa"):
        result["TPSA"]     = round(Descriptors.TPSA(mol), 2)
    if compute_all or kwargs.get("hbd"):
        result["HBD"]      = rdMolDescriptors.CalcNumHBD(mol)
    if compute_all or kwargs.get("hba"):
        result["HBA"]      = rdMolDescriptors.CalcNumHBA(mol)
    if compute_all or kwargs.get("qed"):
        result["QED"]      = round(QED.qed(mol), 4)
    if compute_all or kwargs.get("rings"):
        result["Rings"]    = rdMolDescriptors.CalcNumRings(mol)
    if compute_all or kwargs.get("rotbonds"):
        result["RotBonds"] = rdMolDescriptors.CalcNumRotatableBonds(mol)

    return result

aspirin = "CC(=O)Oc1ccccc1C(=O)O"

# Request specific descriptors
print(compute_descriptors(aspirin, mw=True, logp=True, qed=True))

# Request all
print(compute_descriptors(aspirin, all=True))

---
## Section 5 — Closures and the Factory Pattern

In [ ]:
# ── 5.1 Closures: functions that remember their environment ──────────────────
# A closure is a function that captures variables from its enclosing scope.

def make_ro5_filter(mw_limit=500, logp_limit=5, hbd_limit=5, hba_limit=10):
    """
    Factory: returns a filter function with baked-in limits.
    The returned function 'closes over' mw_limit, logp_limit, etc.
    """
    def filter_fn(smiles: str) -> bool:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return False
        violations = sum([
            Descriptors.MolWt(mol)    > mw_limit,
            Descriptors.MolLogP(mol)  > logp_limit,
            rdMolDescriptors.CalcNumHBD(mol) > hbd_limit,
            rdMolDescriptors.CalcNumHBA(mol) > hba_limit,
        ])
        return violations <= 1

    # The function remembers these values even after make_ro5_filter returns
    filter_fn.__name__ = f"Ro5_mw{mw_limit}_logp{logp_limit}"
    return filter_fn

# Create specialised filters for different use cases
standard_ro5  = make_ro5_filter()                          # Lipinski standard
cns_filter    = make_ro5_filter(mw_limit=400, logp_limit=3.5) # CNS drugs
lead_filter   = make_ro5_filter(mw_limit=350, logp_limit=3)   # Lead-like

library = [
    "CC(=O)Oc1ccccc1C(=O)O",      # Aspirin
    "CC(=O)Nc1ccc(O)cc1",          # Paracetamol
    "CCCC1=NN(C)C(=O)c2cc(cnc21)c1ccc(cc1)S(=O)(=O)N1CCN(C)CC1",  # Sildenafil
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O", # Ibuprofen
]

for name, filt in [("Standard Ro5", standard_ro5), ("CNS", cns_filter), ("Lead-like", lead_filter)]:
    passing = [smi for smi in library if filt(smi)]
    print(f"{name:14s}: {len(passing)}/{len(library)} pass")

In [ ]:
# ── 5.2 Closure for memoisation (manual cache) ───────────────────────────────
def make_cached_descriptor(descriptor_fn):
    """
    Wrap any descriptor function with a SMILES→value cache.
    Useful when the same SMILES appears many times in a large dataset.
    """
    cache = {}  # this dict is captured in the closure

    def cached_fn(smiles: str):
        if smiles not in cache:
            cache[smiles] = descriptor_fn(smiles)
        return cache[smiles]

    cached_fn.cache     = cache          # expose for inspection
    cached_fn.cache_size = lambda: len(cache)
    return cached_fn

def slow_descriptor(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    return round(QED.qed(mol), 4)

fast_qed = make_cached_descriptor(slow_descriptor)

# First call: computed. Second call: from cache.
smiles = "CC(=O)Oc1ccccc1C(=O)O"
import time

t0 = time.perf_counter()
for _ in range(1000): fast_qed(smiles)
t1 = time.perf_counter()

print(f"1000 calls to cached function: {(t1-t0)*1000:.1f} ms")
print(f"Cache size: {fast_qed.cache_size()}")
print(f"Cached value: {fast_qed(smiles)}")

---
## Section 6 — Decorators

In [ ]:
# ── 6.1 Decorator basics ─────────────────────────────────────────────────────
# A decorator is a function that takes a function and returns a new function.
# Syntax: @decorator_name above the function definition.

import functools
import time

def timeit(func):
    """Decorator: print how long a function takes."""    @functools.wraps(func)   # preserves __name__, __doc__ of original
    def wrapper(*args, **kwargs):
        t0     = time.perf_counter()
        result = func(*args, **kwargs)
        dt     = time.perf_counter() - t0
        print(f"[{func.__name__}] elapsed: {dt*1000:.2f} ms")
        return result
    return wrapper

@timeit
def compute_all_qed(smiles_list: list[str]) -> list[float]:
    """Compute QED for a list of SMILES."""    results = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            results.append(QED.qed(mol))
    return results

library = ["CC(=O)Oc1ccccc1C(=O)O"] * 200
scores  = compute_all_qed(library)
print(f"Computed {len(scores)} QED scores")

In [ ]:
# ── 6.2 Practical decorators for scientific code ─────────────────────────────

def validate_smiles(func):
    """Decorator: check that the first argument is a valid SMILES."""    @functools.wraps(func)
    def wrapper(smiles, *args, **kwargs):
        if not isinstance(smiles, str):
            raise TypeError(f"Expected str, got {type(smiles)}")
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError(f"Invalid SMILES: {smiles!r}")
        return func(smiles, *args, **kwargs)
    return wrapper

def returns_none_on_invalid(func):
    """Decorator: return None instead of raising on invalid SMILES."""    @functools.wraps(func)
    def wrapper(smiles, *args, **kwargs):
        try:
            return func(smiles, *args, **kwargs)
        except (ValueError, AttributeError):
            return None
    return wrapper

@validate_smiles
def strict_mw(smiles: str) -> float:
    """Strict version: raises on invalid input."""    return round(Descriptors.MolWt(Chem.MolFromSmiles(smiles)), 2)

@returns_none_on_invalid
@validate_smiles
def safe_mw(smiles: str) -> float | None:
    """Safe version: returns None on invalid input."""    return round(Descriptors.MolWt(Chem.MolFromSmiles(smiles)), 2)

# Stacking decorators: applied bottom-up
print(safe_mw("CC(=O)Oc1ccccc1C(=O)O"))   # 180.16
print(safe_mw("INVALID"))                   # None (safe)
try:
    strict_mw("INVALID")
except ValueError as e:
    print(f"ValueError caught: {e}")        # raises

# functools.lru_cache: built-in memoisation decorator
@functools.lru_cache(maxsize=1024)
def cached_qed(smiles: str) -> float | None:
    """QED with LRU cache — most useful decorator for descriptors."""    mol = Chem.MolFromSmiles(smiles)
    return round(QED.qed(mol), 4) if mol else None

smi = "CC(=O)Oc1ccccc1C(=O)O"
for _ in range(5):
    cached_qed(smi)   # only computed once
print(f"Cache info: {cached_qed.cache_info()}")

---
## Section 7 — Generators and Iterators

In [ ]:
# ── 7.1 Generators: memory-efficient processing of large libraries ────────────
# A generator yields one value at a time — never loads everything into RAM.
# Critical for processing millions of compounds from large SDF files.

def read_smiles_file(filepath: str):
    """
    Generator: yield (smiles, label) one at a time.
    For a 10 million-compound file, this uses ~KB not ~GB.
    """
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split('\t')
            if len(parts) >= 2:
                yield parts[0], parts[1]  # smiles, label
            else:
                yield parts[0], None

def valid_mols_generator(smiles_iterable):
    """Generator: filter and convert to Mol objects lazily."""    for item in smiles_iterable:
        smi = item[0] if isinstance(item, tuple) else item
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            yield mol, smi

# Simulate a large library using a generator expression
smiles_source = [
    ("CC(=O)Oc1ccccc1C(=O)O", "1"),
    ("Cn1cnc2c1c(=O)n(C)c(=O)n2C", "0"),
    ("CC(C)Cc1ccc(cc1)C(C)C(=O)O", "1"),
    ("CC(=O)Nc1ccc(O)cc1", "0"),
    ("INVALID", "0"),
]

# Process without loading all into memory
mw_gen = (
    Descriptors.MolWt(mol)
    for mol, smi in valid_mols_generator(smiles_source)
)

print("Streaming MW computation:")
for i, mw in enumerate(mw_gen):
    print(f"  Compound {i+1}: MW = {mw:.1f}")

In [ ]:
# ── 7.2 Generator with pipeline composition ──────────────────────────────────
# Chain generators to build a preprocessing pipeline — each step is lazy

def parse_smiles(raw_list):
    """Stage 1: parse raw strings."""    for item in raw_list:
        yield item.strip()

def filter_valid(smiles_gen):
    """Stage 2: keep only valid SMILES."""    for smi in smiles_gen:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            yield smi, mol

def compute_qed(mol_gen):
    """Stage 3: compute QED score."""    for smi, mol in mol_gen:
        yield smi, mol, QED.qed(mol)

def filter_by_qed(scored_gen, min_qed=0.5):
    """Stage 4: keep high-QED compounds."""    for smi, mol, qed_val in scored_gen:
        if qed_val >= min_qed:
            yield smi, qed_val

# Compose the pipeline — nothing computed yet!
raw = ["CC(=O)Oc1ccccc1C(=O)O", "INVALID", "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
       "CC(=O)Nc1ccc(O)cc1", "NCCc1ccc(O)c(O)c1"]

pipeline = filter_by_qed(
               compute_qed(
                   filter_valid(
                       parse_smiles(raw))), min_qed=0.4)

# Execution starts HERE when we iterate:
print("High-QED compounds:")
for smi, qed_val in pipeline:
    print(f"  QED={qed_val:.3f}  {smi[:35]}")

---
## Section 8 — Recursive Functions

In [ ]:
# ── 8.1 Recursion in cheminformatics ─────────────────────────────────────────
# Recursion: a function that calls itself on a smaller sub-problem.

def count_ring_atoms(mol, ring_idx: int, visited: set = None) -> int:
    """
    Recursively count atoms in fused ring systems starting from ring_idx.
    Demonstrates recursion on molecular ring data.
    """
    if visited is None:
        visited = set()

    ring_info = mol.GetRingInfo()
    rings     = ring_info.AtomRings()

    if ring_idx >= len(rings) or ring_idx in visited:
        return 0

    visited.add(ring_idx)
    current_atoms = set(rings[ring_idx])
    total         = len(current_atoms)

    # Recurse into adjacent (fused) rings
    for j, other_ring in enumerate(rings):
        if j not in visited and len(current_atoms & set(other_ring)) >= 2:
            total += count_ring_atoms(mol, j, visited)

    return total

# Naphthalene (2 fused 6-membered rings)
mol = Chem.MolFromSmiles("c1ccc2ccccc2c1")
print(f"Naphthalene ring atoms: {count_ring_atoms(mol, 0)}")  # 10

# Anthracene (3 fused rings)
mol = Chem.MolFromSmiles("c1ccc2cc3ccccc3cc2c1")
print(f"Anthracene ring atoms:  {count_ring_atoms(mol, 0)}")  # 14

In [ ]:
# ── 8.2 Recursive combinatorial enumeration ──────────────────────────────────
# Enumerate all possible R-group combinations for a scaffold (combinatorial library)

def enumerate_library(scaffold_parts: list[list[str]],
                      current: list = None) -> list[list[str]]:
    """
    Recursively generate all combinations from a list of R-group options.
    scaffold_parts = [[R1_options], [R2_options], [R3_options]]
    """
    if current is None:
        current = []

    if len(current) == len(scaffold_parts):
        return [current.copy()]   # base case: all positions filled

    pos      = len(current)
    combos   = []
    for option in scaffold_parts[pos]:
        current.append(option)
        combos.extend(enumerate_library(scaffold_parts, current))
        current.pop()

    return combos

# Three R-group positions
r_groups = [
    ["Cl", "F", "OMe"],          # R1: halogen or methoxy
    ["H", "Me", "Et"],           # R2: alkyl
    ["OH", "NH2", "COOH"],       # R3: polar
]

library = enumerate_library(r_groups)
print(f"Combinatorial library size: {len(library)}  ({3*3*3} expected)")
print("First 5 combinations:")
for combo in library[:5]:
    print(f"  R1={combo[0]:4s}  R2={combo[1]:4s}  R3={combo[2]}")

---
## Section 9 — Functional Pipelines for QSAR

In [ ]:
# ── 9.1 Full preprocessing pipeline using functional style ───────────────────
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.FilterCatalog import FilterCatalogParams, FilterCatalog

# Build PAINS filter once (expensive to create)
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
PAINS_CATALOG = FilterCatalog(params)

# Each step is a pure function: input → output, no side effects

def step_parse(smiles: str) -> Chem.Mol | None:
    return Chem.MolFromSmiles(smiles.strip()) if smiles else None

def step_largest_fragment(mol: Chem.Mol) -> Chem.Mol:
    return rdMolStandardize.LargestFragmentChooser().choose(mol)

def step_neutralize(mol: Chem.Mol) -> Chem.Mol:
    return rdMolStandardize.Uncharger().uncharge(mol)

def step_ro5(mol: Chem.Mol) -> Chem.Mol | None:
    mw   = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd  = rdMolDescriptors.CalcNumHBD(mol)
    hba  = rdMolDescriptors.CalcNumHBA(mol)
    return mol if sum([mw>500, logp>5, hbd>5, hba>10]) <= 1 else None

def step_pains(mol: Chem.Mol) -> Chem.Mol | None:
    return None if PAINS_CATALOG.GetFirstMatch(mol) else mol

def step_to_canonical(mol: Chem.Mol) -> str:
    return Chem.MolToSmiles(mol)

# Pipeline runner: apply steps in order, short-circuit on None
def run_pipeline(smiles: str, steps: list) -> str | None:
    result = smiles
    for step in steps:
        if result is None:
            return None
        result = step(result)
    return result

PIPELINE = [step_parse, step_largest_fragment, step_neutralize,
            step_ro5, step_pains, step_to_canonical]

test_cases = [
    "[Na+].[O-]C(=O)c1ccccc1",            # salt → strip
    "CC(=O)Oc1ccccc1C(=O)O",              # clean aspirin
    "O=C1CSC(=S)N1",                       # PAINS (rhodanine)
    "C" * 60,                              # MW >> 500: Ro5 fail
]

print("Pipeline results:")
for smi in test_cases:
    result = run_pipeline(smi, PIPELINE)
    status = "✓ PASS" if result else "✗ FAIL"
    print(f"  {status}  {smi[:40]:40s}  →  {result if result else 'REMOVED'}")

---
## Section 10 — NumPy Vectorised Functions

In [ ]:
# ── 10.1 Vectorisation: do it to the whole array at once ─────────────────────
# NumPy operations on arrays are orders of magnitude faster than Python loops.
import numpy as np

# Example: Tanimoto similarity using NumPy (pure vectorised)
def tanimoto_matrix(X: np.ndarray) -> np.ndarray:
    """
    Compute full N×N Tanimoto similarity matrix for a binary fingerprint matrix.
    X shape: [N_molecules, N_bits]
    Uses vectorised dot products — much faster than nested loops.
    """
    # Number of ON bits per row (sum of binary vector)
    counts = X.sum(axis=1, keepdims=True)          # [N, 1]
    # Intersection: X @ X.T gives number of shared ON bits
    intersection = X @ X.T                          # [N, N]
    # Union = count_i + count_j - intersection
    union = counts + counts.T - intersection        # [N, N]
    # Avoid division by zero
    return np.where(union > 0, intersection / union, 0.0)

# Simulate fingerprint matrix (200 compounds × 1024 bits)
np.random.seed(42)
fps = (np.random.rand(200, 1024) > 0.85).astype(np.float32)  # ~15% density

import time
t0  = time.perf_counter()
sim = tanimoto_matrix(fps)
t1  = time.perf_counter()

print(f"Tanimoto matrix {sim.shape}: {(t1-t0)*1000:.1f} ms")
print(f"Mean similarity (off-diagonal): {sim[sim < 1].mean():.4f}")

# Compare with loop (slow)
t0 = time.perf_counter()
slow = np.zeros((10, 10))
for i in range(10):
    for j in range(10):
        shared = (fps[i] * fps[j]).sum()
        union  = fps[i].sum() + fps[j].sum() - shared
        slow[i,j] = shared/union if union > 0 else 0
t1 = time.perf_counter()
print(f"Loop (10×10 only):  {(t1-t0)*1000:.1f} ms  ← ~1000× slower per element")

In [ ]:
# ── 10.2 np.vectorize and apply_along_axis ───────────────────────────────────
# When you can't avoid calling a Python function per element

def lipinski_violations_scalar(mw: float, logp: float,
                                hbd: int, hba: int) -> int:
    """Count Ro5 violations for one compound."""    return int(mw > 500) + int(logp > 5) + int(hbd > 5) + int(hba > 10)

# np.vectorize: apply scalar function to every row
vec_violations = np.vectorize(lipinski_violations_scalar)

# Fake descriptor matrix for 5 compounds
props = np.array([
    [180.0, 1.2, 1, 3],   # Aspirin
    [194.0, -0.1, 1, 6],  # Caffeine
    [206.0, 3.97, 1, 2],  # Ibuprofen
    [151.0, 0.46, 2, 3],  # Paracetamol
    [666.0, 8.5, 3, 12],  # Hypothetical bad compound
])

violations = vec_violations(props[:,0], props[:,1],
                             props[:,2].astype(int), props[:,3].astype(int))
print("Ro5 violations:", violations)
print("Passes Ro5 (≤1 violation):", violations <= 1)

# apply_along_axis: apply function to each row/column
def zscore_row(row: np.ndarray) -> np.ndarray:
    """Z-score normalise one row."""    mu, sigma = row.mean(), row.std()
    return (row - mu) / sigma if sigma > 0 else row - mu

normalised = np.apply_along_axis(zscore_row, axis=1, arr=props)
print(f"\nZ-score normalised shape: {normalised.shape}")
print(f"First row mean≈0: {normalised[0].mean():.6f}")

---
## Section 11 — SciPy for Toxicology

In [ ]:
# ── 11.1 Dose-response curve fitting ─────────────────────────────────────────
# The most important calculation in toxicology: fitting a Hill equation to data
from scipy.optimize import curve_fit
from scipy.stats import ttest_ind, mannwhitneyu, pearsonr, spearmanr
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

def hill_equation(concentration: np.ndarray,
                  ec50: float, hill_coeff: float,
                  top: float, bottom: float) -> np.ndarray:
    """
    4-parameter Hill / logistic equation for dose-response.

    response = bottom + (top - bottom) / (1 + (EC50/C)^n)

    Parameters
    ----------
    concentration : array of concentrations (μM)
    ec50          : concentration at 50% response
    hill_coeff    : Hill coefficient (steepness)
    top, bottom   : upper and lower asymptotes (% effect)
    """
    return bottom + (top - bottom) / (1 + (ec50 / concentration) ** hill_coeff)

# Simulate hERG dose-response data (typical assay data)
np.random.seed(42)
concs    = np.array([0.001, 0.01, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0])
true_ec50, true_n, true_top, true_bot = 1.5, 1.2, 95.0, 2.0
response = hill_equation(concs, true_ec50, true_n, true_top, true_bot)
noisy    = response + np.random.normal(0, 3, len(concs))  # add measurement noise

# Fit the curve using non-linear least squares
try:
    popt, pcov = curve_fit(
        hill_equation, concs, noisy,
        p0=[1.0, 1.0, 100.0, 0.0],          # initial guesses
        bounds=([0.0001, 0.1, 50.0, -10.0],  # lower bounds
                [1000.0, 5.0, 120.0, 20.0]), # upper bounds
        maxfev=10000
    )
    perr = np.sqrt(np.diag(pcov))  # standard errors
    ec50_fit, n_fit, top_fit, bot_fit = popt

    print(f"Fitted parameters:")
    print(f"  EC50 = {ec50_fit:.3f} μM  (true: {true_ec50})  ±{perr[0]:.3f}")
    print(f"  Hill = {n_fit:.3f}       (true: {true_n})   ±{perr[1]:.3f}")
    print(f"  Top  = {top_fit:.1f}%      (true: {true_top})  ±{perr[2]:.2f}")

except RuntimeError as e:
    print(f"Curve fit failed: {e}")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
concs_fine = np.logspace(-3, 2, 200)
ax.semilogx(concs, noisy, 'o', color='#1565C0', ms=8, label='Observed data')
ax.semilogx(concs_fine, hill_equation(concs_fine, *popt),
            '-', color='#E74C3C', lw=2.5, label=f'Fit  EC50={ec50_fit:.2f} μM')
ax.axvline(ec50_fit, color='#E74C3C', linestyle='--', alpha=0.5)
ax.axhline(50, color='grey', linestyle=':', alpha=0.5)
ax.set_xlabel('Concentration (μM)', fontsize=12)
ax.set_ylabel('% Inhibition', fontsize=12)
ax.set_title('hERG IC50 Dose-Response Curve', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── 11.2 Statistical tests in toxicology ─────────────────────────────────────
# Comparing control vs treated groups — standard in in vitro tox assays

from scipy.stats import ttest_ind, mannwhitneyu, kruskal, shapiro, f_oneway

np.random.seed(42)

# Simulate cell viability data (% of control)
ctrl   = np.random.normal(100, 8, 24)    # control: 24 replicates
low    = np.random.normal(92,  9, 24)    # low dose: slight toxicity
mid    = np.random.normal(71, 10, 24)    # mid dose: moderate
high   = np.random.normal(38, 12, 24)    # high dose: severe

def test_group(name: str, group: np.ndarray, ctrl: np.ndarray):
    # Test normality first
    _, p_norm = shapiro(group)
    normal = p_norm > 0.05

    if normal:
        # Parametric: Student's t-test
        _, p_val = ttest_ind(ctrl, group)
        test_name = "t-test"
    else:
        # Non-parametric: Mann-Whitney U
        _, p_val = mannwhitneyu(ctrl, group, alternative='greater')
        test_name = "Mann-Whitney"

    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
    print(f"  {name:10s}: mean={group.mean():5.1f}%  {test_name}  p={p_val:.4f}  {sig}")

print("Cell viability vs control:")
for name, grp in [("Low dose", low), ("Mid dose", mid), ("High dose", high)]:
    test_group(name, grp, ctrl)

# One-way ANOVA across all groups
_, p_anova = f_oneway(ctrl, low, mid, high)
print(f"\nOne-way ANOVA (all groups): p = {p_anova:.2e}")
print("→ Significant overall effect" if p_anova < 0.05 else "→ No significant effect")

---
## Section 12 — Error Handling in Scientific Code

In [ ]:
# ── 12.1 Custom exceptions for scientific validation ─────────────────────────

class InvalidSMILESError(ValueError):
    """Raised when a SMILES string cannot be parsed by RDKit."""    def __init__(self, smiles: str, message: str = ""):
        self.smiles = smiles
        super().__init__(f"Invalid SMILES {smiles!r}" + (f": {message}" if message else ""))

class DescriptorError(RuntimeError):
    """Raised when descriptor calculation fails unexpectedly."""    pass

class ApplicabilityDomainError(Warning):
    """Raised when a compound falls outside the model applicability domain."""    pass

def safe_descriptor_calc(smiles: str,
                          descriptor_fn,
                          default=None,
                          warn_ad: bool = True):
    """
    Production-grade descriptor calculation:
      - Validates input
      - Catches specific exceptions
      - Returns default on failure
      - Warns on AD violations
    """
    if not isinstance(smiles, str) or not smiles.strip():
        return default

    try:
        mol = Chem.MolFromSmiles(smiles.strip())
        if mol is None:
            raise InvalidSMILESError(smiles)

        result = descriptor_fn(mol)

        # Applicability domain check (simplified: MW > 1000 is suspicious)
        if warn_ad and Descriptors.MolWt(mol) > 800:
            import warnings
            warnings.warn(f"MW > 800 Da: compound may be outside AD", ApplicabilityDomainError)

        return result

    except InvalidSMILESError:
        return default
    except Exception as e:
        raise DescriptorError(f"Descriptor failed for {smiles}: {e}") from e

# Test
import warnings
print(safe_descriptor_calc("CC(=O)Oc1ccccc1C(=O)O",  QED.qed))    # works
print(safe_descriptor_calc("INVALID",                  QED.qed))    # returns None
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    result = safe_descriptor_calc("C" * 200 + "c1ccccc1", QED.qed)  # triggers AD warning
    if w:
        print(f"Warning: {w[0].message}")

In [ ]:
# ── 12.2 Cheatsheet ──────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║        Python Functions — Scientific Computing Quick Reference          ║
╠══════════════════════════════════════════════════════════════════════════╣
║ SYNTAX                                                                   ║
║  def fn(pos, /, std, *, kw_only):              positional-only /       ║
║  def fn(*args, **kwargs):                       variadic                ║
║  fn = lambda x: x**2                           anonymous                ║
║  @decorator                                     wraps function           ║
║  @functools.lru_cache(maxsize=1024)             memoisation              ║
╠══════════════════════════════════════════════════════════════════════════╣
║ PATTERNS                                                                 ║
║  Factory     make_filter(limit) → filter_fn    bake-in parameters       ║
║  Closure     cache = {}; def fn(): ...         stateful function         ║
║  Generator   def gen(): yield item             lazy evaluation           ║
║  Pipeline    reduce(compose, [f1, f2, f3])     functional chain         ║
║  Vectorise   np.vectorize(fn)(array)           apply to all elements    ║
╠══════════════════════════════════════════════════════════════════════════╣
║ SCIENTIFIC IDIOMS                                                        ║
║  # Safe SMILES                                                           ║
║  mol = Chem.MolFromSmiles(smi); assert mol is not None                  ║
║  # Curve fitting                                                         ║
║  popt, pcov = scipy.optimize.curve_fit(fn, x, y, p0=[...])             ║
║  # Dose-response EC50                                                    ║
║  def hill(c, ec50, n, top, bot): ...                                    ║
║  # Vectorised Tanimoto                                                   ║
║  sim = (X @ X.T) / (a + a.T - X @ X.T)                                ║
╠══════════════════════════════════════════════════════════════════════════╣
║ GOLDEN RULES                                                             ║
║  1. One function = one responsibility                                    ║
║  2. Pure functions (no side effects) are easier to test                 ║
║  3. Use generators for large datasets (memory efficiency)               ║
║  4. lru_cache any pure function called repeatedly                       ║
║  5. Type hints + docstrings for all public functions                    ║
║  6. Handle None from MolFromSmiles before every descriptor call         ║
╚══════════════════════════════════════════════════════════════════════════╝
""")